# Proyecto Big Data – Procesamiento distribuido de GeoNames en GCP
**Universidad de Ingeniería y Tecnología – UTEC** · Facultad de Computación · Curso: Big Data (DS4341)

**Integrantes:** Daniel Guillermo Sandoval Toro · Elizabeth Huamán Santillán · Estefano Mauricio Zarate Manosalva · Matías Castillo Quincho · Salvador Samuel Olivares Leandro

Notebook único y autocontenido del proyecto. Incluye:

1. Configuración del entorno
2. Ingesta batch del dataset al Data Lake (Google Cloud Storage)
3. Las mismas 10 consultas con **Polars**, **Dask**, **Modin** y **Apache Spark**
4. Benchmark y verificación de resultados
5. Evidencias de GCP (bucket y clúster Dataproc)
6. **Hadoop MapReduce** en Dataproc con HDFS (`wordmean`, `wordmedian`, `secondarysort`)
7. Indicadores para la toma de decisiones

> **Dónde ejecutarlo:** en el **Jupyter del clúster Dataproc** (Dataproc → Clústeres → `cluster-proyecto-bigdata`
> → Interfaces web → Jupyter), porque ahí están `hdfs` y `hadoop` para la parte de MapReduce y Spark corre sobre YARN.
> Usar el kernel **Python 3**. Las secciones 1 a 4 también funcionan en Colab o en cualquier máquina con Python.
>
> **Si el clúster no tiene salida a internet** (solo IPs internas y sin Cloud NAT), `%pip install` y la
> descarga de GeoNames fallarán. En ese caso: correr las secciones 1–4 en Colab (o en una VM con internet)
> y las secciones 5–6 en el Jupyter de Dataproc, ejecutando antes las celdas de configuración de la sección 1.

## Dataset
**Origen:** GeoNames (https://download.geonames.org/export/dump/allCountries.zip), base geográfica mundial
de libre uso (CC BY 4.0), actualizada a diario.

**Estructura:** TSV sin encabezado → CSV con encabezados. **13,464,034 registros × 19 variables (≈ 1.8 GB)**.

| Variable | Tipo | Descripción |
|---|---|---|
| geonameid | int64 | Identificador único |
| name / asciiname | string | Nombre (UTF-8 / ASCII) |
| alternatenames | string | Nombres alternativos separados por coma (semiestructurado) |
| latitude / longitude | float64 | Coordenadas WGS84 |
| fclass / fcode | string | Clase y código de entidad (P poblado, H hidrografía, T relieve, …) |
| country / cc2 | string | País ISO-3166 alfa-2 / códigos alternativos |
| admin1…admin4 | string | Divisiones administrativas |
| population | int64 | Población |
| elevation | int | Elevación en metros (mayormente vacía) |
| dem | int | Modelo digital de elevación (−9999 = sin dato) |
| timezone | string | Zona horaria IANA |
| moddate | date | Fecha de última modificación |

## 1. Configuración del entorno

In [ ]:
# Instalación de dependencias (en Dataproc/Colab solo faltan algunas; tarda ~1-2 min)
%pip install -q polars "dask[dataframe]" "modin[ray]" pyarrow psutil matplotlib

In [ ]:
import os, sys, gc, json, time, shutil, hashlib, threading, subprocess, random
from contextlib import contextmanager
import pandas as pd
import psutil
import matplotlib.pyplot as plt
from IPython.display import display

# ---------------------------------------------------------------- parámetros
BUCKET        = "utec-bigdata-geonames-2026"
GCS_RAW       = f"gs://{BUCKET}/raw/allCountries_headers.csv"
GCS_PROCESSED = f"gs://{BUCKET}/processed"
REGION, ZONA, CLUSTER = "us-east4", "us-east4-b", "cluster-proyecto-bigdata"

DATA_DIR = os.path.expanduser("~/geonames")                 # copia local del dataset
DATA     = os.environ.get("DATA_PATH", f"{DATA_DIR}/allCountries_headers.csv")
DATA_3M  = os.environ.get("DATA_3M_PATH", f"{DATA_DIR}/muestra_3M.csv")
OUT      = "results"                                          # resultados de cada consulta

EJECUTAR_INGESTA   = False   # True solo la 1.ª vez: descarga GeoNames y sube el CSV completo al bucket
EJECUTAR_MAPREDUCE = True    # sección 6 (requiere estar en el master de Dataproc)
EJECUTAR_TERASORT  = False   # extra opcional (≈ 1 GB en HDFS)
ESCRIBIR_PROCESSED = True    # Spark escribe Parquet en gs://.../processed/ (solo en Dataproc)

EN_DATAPROC = shutil.which("hdfs") is not None
RAM_GB = psutil.virtual_memory().total / 1e9
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(f"Dataproc: {EN_DATAPROC} | RAM: {RAM_GB:.1f} GB | CPUs: {os.cpu_count()} | dataset: {DATA}")

In [ ]:
# ---------------------------------------------------------------- esquema y constantes
ALL_COLUMNS = ["geonameid", "name", "asciiname", "alternatenames", "latitude", "longitude",
               "fclass", "fcode", "country", "cc2", "admin1", "admin2", "admin3", "admin4",
               "population", "elevation", "dem", "timezone", "moddate"]

# Proyección de columnas: se descartan alternatenames, asciiname, cc2 y admin2-4 (no se usan y
# son las columnas de texto más pesadas) -> menos de la mitad de memoria.
USE_COLUMNS = ["geonameid", "name", "latitude", "longitude", "fclass", "fcode", "country",
               "admin1", "population", "elevation", "dem", "timezone", "moddate"]

PANDAS_DTYPES = {  # tipos explícitos para Dask y Modin (evita DtypeWarning y ahorra memoria)
    "geonameid": "int64", "name": "string[pyarrow]", "latitude": "float64", "longitude": "float64",
    "fclass": "category", "fcode": "category", "country": "category", "admin1": "string[pyarrow]",
    "population": "float64", "elevation": "float64", "dem": "float64", "timezone": "category",
    "moddate": "string[pyarrow]"}

FCLASS_DESC = {"A": "Division administrativa", "H": "Hidrografia (rios, lagos)", "L": "Areas y parques",
               "P": "Ciudades y poblados", "R": "Carreteras y vias ferreas",
               "S": "Edificios y puntos de interes", "T": "Relieve (montanas, cerros)",
               "U": "Submarino", "V": "Vegetacion y bosques"}
POP_BINS   = [float("-inf"), 1, 1_000, 100_000, 1_000_000, float("inf")]   # intervalos [a, b)
POP_LABELS = ["Sin poblacion", "< 1K", "1K - 100K", "100K - 1M", "> 1M"]
DUP_KEYS   = ["name", "country", "latitude", "longitude"]   # duplicado lógico
SOUTH_AMERICA = ["AR", "BO", "BR", "CL", "CO", "EC", "GY", "PE", "PY", "SR", "UY", "VE"]

QUERIES = {
    "Q1": "Limpieza y tipado: nulos por columna, casteo, trim, DEM -9999 -> nulo",
    "Q2": "Duplicados por geonameid y por nombre+país+coordenadas (se conserva el menor id)",
    "Q3": "Validación y eliminación (DELETE) de registros inválidos",
    "Q4": "Transformación de variables (CREATE/UPDATE)",
    "Q5": "Filtrado: centros poblados del Perú ordenados por población",
    "Q6": "Agregación por país: registros, población de asentamientos, elevación promedio",
    "Q7": "Agrupación por clase de entidad (fclass) con % del total",
    "Q8": "Ciudades con más de 1 millón de habitantes por país",
    "Q9": "Top 3 centros poblados por país de Sudamérica (ranking)",
    "Q10": "Registros modificados por año y % acumulado"}

In [ ]:
# ---------------------------------------------------------------- utilidades
TIEMPOS = {}     # {framework: {paso: segundos}}
RAM_PICO = {}    # {framework: GB}

class Medidor:
    """Mide el tiempo de cada paso y la RAM pico (proceso + hijos: workers de Ray, JVM de Spark)."""
    def __init__(self, fw):
        self.fw, self.pico, self._run = fw, 0, True
        TIEMPOS[fw] = {}
        threading.Thread(target=self._muestrear, daemon=True).start()
    def _muestrear(self):
        yo = psutil.Process()
        while self._run:
            total = 0
            for p in [yo] + yo.children(recursive=True):
                try:
                    m = p.memory_info(); total += m.rss - getattr(m, "shared", 0)
                except psutil.Error:
                    pass
            self.pico = max(self.pico, total); time.sleep(0.25)
    @contextmanager
    def medir(self, paso):
        if paso in QUERIES:
            print(f"{paso} - {QUERIES[paso]}")
        t0 = time.perf_counter(); yield
        TIEMPOS[self.fw][paso] = round(time.perf_counter() - t0, 3)
        print(f"[{self.fw}] {paso}: {TIEMPOS[self.fw][paso]:.3f} s")
    def cerrar(self):
        self._run = False
        TIEMPOS[self.fw]["total_consultas"] = round(sum(v for k, v in TIEMPOS[self.fw].items() if k.startswith("Q")), 3)
        RAM_PICO[self.fw] = round(self.pico / 1e9, 2)
        print(f"[{self.fw}] 10 consultas: {TIEMPOS[self.fw]['total_consultas']} s | RAM pico: {RAM_PICO[self.fw]} GB")

def guardar(pdf, fw, q, n=10):
    """Guarda el resultado (pandas) en results/<fw>/<q>.csv y lo muestra."""
    os.makedirs(f"{OUT}/{fw}", exist_ok=True)
    pdf.to_csv(f"{OUT}/{fw}/{q}.csv", index=False)
    display(pdf.head(n))

def sh(cmd):
    """Ejecuta un comando de shell mostrando el comando y su salida en la celda."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for linea in p.stdout:
        print(linea, end="")
    rc = p.wait()
    if rc:
        print(f"(código de salida {rc})")

## 2. Ingesta batch al Data Lake (Cloud Storage)
Flujo: **GeoNames (HTTP) → conversión TSV→CSV con validación → `gs://utec-bigdata-geonames-2026/raw/`**.

> El CSV del primer avance estaba **truncado** (10,330,862 filas; se cortaba en Tailandia). Con
> `EJECUTAR_INGESTA = True` se regenera el CSV completo y se **sobrescribe** el del bucket.

In [ ]:
def preparar_dataset(entrada_zip, salida_csv):
    """allCountries.txt (TSV sin encabezado, dentro del ZIP) -> CSV con encabezados, en streaming."""
    import csv, io, zipfile
    csv.field_size_limit(10**9)
    n = 0
    with zipfile.ZipFile(entrada_zip) as z, \
         io.TextIOWrapper(z.open("allCountries.txt"), encoding="utf-8", newline="") as fin, \
         open(salida_csv, "w", encoding="utf-8", newline="") as fout:
        escritor = csv.writer(fout)
        escritor.writerow(ALL_COLUMNS)
        for fila in csv.reader(fin, delimiter="\t", quoting=csv.QUOTE_NONE):
            if len(fila) != len(ALL_COLUMNS):
                raise ValueError(f"Fila {n + 1} con {len(fila)} columnas")
            escritor.writerow(fila); n += 1
    print(f"{n:,} registros escritos en {salida_csv}")
    return n

if EJECUTAR_INGESTA:
    zip_local = f"{DATA_DIR}/allCountries.zip"
    sh(f"wget -q -O {zip_local} https://download.geonames.org/export/dump/allCountries.zip")
    n = preparar_dataset(zip_local, DATA)
    assert n == 13_464_034, "El conteo no coincide con el dump original"
    sh(f"gcloud storage cp {zip_local} gs://{BUCKET}/raw/source/allCountries.zip")
    sh(f"gcloud storage cp {DATA} {GCS_RAW}")
elif not os.path.exists(DATA):
    # descarga la capa RAW del bucket al disco local (lectura más rápida para Polars/Dask/Modin)
    sh(f"gcloud storage cp {GCS_RAW} {DATA}")

In [ ]:
# Validación del volumen: 13,464,034 registros + 1 línea de encabezado
sh(f"ls -lh {DATA}")
with open(DATA, "rb") as f:
    n_lineas = sum(1 for _ in f)
N_TOTAL = n_lineas - 1
print(f"Registros: {N_TOTAL:,}")
if N_TOTAL != 13_464_034:
    print("ATENCIÓN: el archivo no tiene los 13,464,034 registros del dump completo (¿CSV truncado?)")

In [ ]:
# Muestra aleatoria de 3 M de registros (semilla 42): se usa para Modin si la máquina tiene < 16 GB de RAM
if not os.path.exists(DATA_3M):
    random.seed(42)
    idx = set(random.sample(range(N_TOTAL), min(3_000_000, N_TOTAL)))
    with open(DATA, encoding="utf-8") as f, open(DATA_3M, "w", encoding="utf-8") as o:
        o.write(f.readline())
        for i, linea in enumerate(f):
            if i in idx:
                o.write(linea)
print("muestra lista:", DATA_3M)

## 3. Consultas implementadas (idénticas en los 4 frameworks)
| # | Consulta | Operación de la rúbrica |
|---|---|---|
| Q1 | Nulos por columna, casteo de tipos, trim de nombres, población nula → 0, DEM −9999 → nulo | Limpieza, validación, UPDATE |
| Q2 | Duplicados por `geonameid` y duplicados lógicos (nombre+país+coordenadas) | Tratamiento de duplicados, DELETE |
| Q3 | Reglas: lat ∈ [−90, 90], lon ∈ [−180, 180], población ≥ 0, país y nombre no nulos | Validación, DELETE |
| Q4 | Nuevas columnas `anio_mod`, `hemisferio`, `elevacion_final`, `poblacion_asentamiento`, `fclass_desc`, `categoria_poblacion` | Transformación de variables, CREATE |
| Q5 | Centros poblados del Perú con población, ordenados | Filtrado, ordenamiento |
| Q6 | Por país: registros, población (solo `fclass = P`) y elevación promedio | Agregación, agrupación |
| Q7 | Registros por clase de entidad y % | Agrupación, métrica |
| Q8 | Ciudades con más de 1 M de habitantes por país | Filtrado, agrupación |
| Q9 | Top 3 ciudades por país de Sudamérica | Ranking (ventana), ordenamiento |
| Q10 | Registros modificados por año y % acumulado | Métrica temporal |

## 3.1 Polars
Motor columnar en Rust (Apache Arrow), **multihilo**, con API *lazy*: `scan_csv` construye un plan que se optimiza
(proyección de columnas, *predicate pushdown*) antes de ejecutarse en paralelo en todos los núcleos.

In [ ]:
import polars as pl
M = Medidor("polars")
SCHEMA_PL = {"geonameid": pl.Int64, "name": pl.Utf8, "latitude": pl.Float64, "longitude": pl.Float64,
             "fclass": pl.Utf8, "fcode": pl.Utf8, "country": pl.Utf8, "admin1": pl.Utf8,
             "population": pl.Int64, "elevation": pl.Float64, "dem": pl.Int64,
             "timezone": pl.Utf8, "moddate": pl.Utf8}
with M.medir("carga"):
    pdf_ = (pl.scan_csv(DATA, schema_overrides=SCHEMA_PL, infer_schema_length=0)
              .select(USE_COLUMNS)
              .with_columns(pl.col("fclass", "fcode", "country", "timezone").cast(pl.Categorical))
              .collect())
df = pdf_; del pdf_
print(f"{df.height:,} filas x {df.width} columnas")
df.head()

In [ ]:
with M.medir("Q1"):   # Limpieza y tipado
    nulos = (df.null_count().transpose(include_header=True, column_names=["nulos"])
               .rename({"column": "columna"}))
    df = df.with_columns(
        pl.col("name").str.strip_chars(),
        pl.col("population").fill_null(0),
        pl.when(pl.col("dem") == -9999).then(None).otherwise(pl.col("dem")).alias("dem"),
        pl.col("moddate").str.to_date("%Y-%m-%d", strict=False))
guardar(nulos.to_pandas(), "polars", "Q1", n=20)

In [ ]:
with M.medir("Q2"):   # Duplicados
    n0 = df.height
    dup_id = n0 - df.select(pl.col("geonameid").n_unique()).item()
    df = df.unique(subset=["geonameid"], keep="first")
    n1 = df.height
    # duplicados lógicos: hash de 64 bits de la clave compuesta; se conserva el menor geonameid
    clave = pl.struct(DUP_KEYS).hash()
    en_dup = df.filter(clave.is_duplicated()).select(*DUP_KEYS, "geonameid")
    eliminar = en_dup.filter(pl.col("geonameid") != pl.col("geonameid").min().over(DUP_KEYS))
    df = df.filter(~pl.col("geonameid").is_in(eliminar.get_column("geonameid").implode()))
    q2 = pd.DataFrame({"criterio": ["geonameid", "name+country+lat+lon", "registros_finales"],
                       "valor": [dup_id, n1 - df.height, df.height]})
guardar(q2, "polars", "Q2")

In [ ]:
with M.medir("Q3"):   # Validación y DELETE
    reglas = {"latitud_fuera_rango": ~pl.col("latitude").is_between(-90, 90),
              "longitud_fuera_rango": ~pl.col("longitude").is_between(-180, 180),
              "poblacion_negativa": pl.col("population") < 0,
              "pais_nulo": pl.col("country").is_null(),
              "nombre_nulo": pl.col("name").is_null()}
    conteos = df.select([e.fill_null(False).sum().alias(k) for k, e in reglas.items()]).row(0)
    antes = df.height
    df = df.filter(~pl.any_horizontal([e.fill_null(False) for e in reglas.values()]))
    q3 = pd.DataFrame({"regla": list(reglas) + ["TOTAL_ELIMINADOS"],
                       "registros": list(conteos) + [antes - df.height]})
guardar(q3, "polars", "Q3")

In [ ]:
with M.medir("Q4"):   # Transformación de variables (CREATE / UPDATE)
    pop = pl.col("population")
    df = df.with_columns(
        pl.col("moddate").dt.year().alias("anio_mod"),
        pl.when(pl.col("latitude") >= 0).then(pl.lit("Norte")).otherwise(pl.lit("Sur")).alias("hemisferio"),
        pl.coalesce(pl.col("elevation"), pl.col("dem").cast(pl.Float64)).alias("elevacion_final"),
        pl.when(pl.col("fclass") == "P").then(pop).otherwise(0).alias("poblacion_asentamiento"),
        pl.col("fclass").cast(pl.Utf8).replace_strict(FCLASS_DESC, default="Desconocido").alias("fclass_desc"),
        pl.when(pop <= 0).then(pl.lit(POP_LABELS[0])).when(pop < 1_000).then(pl.lit(POP_LABELS[1]))
          .when(pop < 100_000).then(pl.lit(POP_LABELS[2])).when(pop < 1_000_000).then(pl.lit(POP_LABELS[3]))
          .otherwise(pl.lit(POP_LABELS[4])).alias("categoria_poblacion"))
    q4 = df.group_by("categoria_poblacion").len("registros").sort("registros", descending=True)
guardar(q4.to_pandas(), "polars", "Q4")

In [ ]:
with M.medir("Q5"):   # Filtrado + ordenamiento
    q5 = (df.filter((pl.col("country") == "PE") & (pl.col("fclass") == "P") & (pl.col("population") > 0))
            .sort("population", descending=True)
            .select("name", "admin1", "population", "elevacion_final", "latitude", "longitude"))
print(f"Centros poblados con población en Perú: {q5.height:,}")
guardar(q5.head(20).to_pandas(), "polars", "Q5")

In [ ]:
with M.medir("Q6"):   # Agregación por país
    q6 = (df.group_by("country")
            .agg(pl.len().alias("registros"),
                 pl.col("poblacion_asentamiento").sum().alias("poblacion_total"),
                 pl.col("elevacion_final").mean().round(2).alias("elevacion_promedio"))
            .sort("poblacion_total", descending=True))
guardar(q6.to_pandas(), "polars", "Q6")

In [ ]:
with M.medir("Q7"):   # Agrupación por clase de entidad
    total = df.height
    q7 = (df.group_by("fclass", "fclass_desc").len("registros")
            .with_columns((pl.col("registros") / total * 100).round(2).alias("porcentaje"))
            .sort("registros", descending=True))
guardar(q7.to_pandas(), "polars", "Q7", n=12)

In [ ]:
with M.medir("Q8"):   # Ciudades de más de 1 M
    q8 = (df.filter((pl.col("fclass") == "P") & (pl.col("population") > 1_000_000))
            .group_by("country")
            .agg(pl.len().alias("ciudades_mas_1M"), pl.col("population").sum().alias("poblacion_en_esas_ciudades"))
            .sort(["ciudades_mas_1M", "poblacion_en_esas_ciudades"], descending=[True, True]))
guardar(q8.to_pandas(), "polars", "Q8")

In [ ]:
with M.medir("Q9"):   # Ranking por ventana
    q9 = (df.filter(pl.col("country").cast(pl.Utf8).is_in(SOUTH_AMERICA) & (pl.col("fclass") == "P"))
            .with_columns(pl.col("population").rank("ordinal", descending=True).over("country").alias("ranking"))
            .filter(pl.col("ranking") <= 3)
            .select("country", "ranking", "name", "population")
            .sort(["country", "ranking"]))
guardar(q9.to_pandas(), "polars", "Q9", n=36)

In [ ]:
with M.medir("Q10"):  # Métrica temporal
    q10 = (df.group_by("anio_mod").len("registros").drop_nulls("anio_mod").sort("anio_mod")
             .with_columns((pl.col("registros").cum_sum() / pl.col("registros").sum() * 100)
                           .round(2).alias("pct_acumulado")))
guardar(q10.to_pandas(), "polars", "Q10", n=40)
M.cerrar()

In [ ]:
# Liberar memoria antes del siguiente framework
del df, q5, q6, q7, q8, q9, q10, en_dup, eliminar
gc.collect();

## 3.2 Dask
El CSV se divide en **particiones de 64 MB** (un DataFrame pandas por partición) y se construye un **grafo de
tareas (DAG)** perezoso que el *scheduler* ejecuta en paralelo. `persist()` cachea el DataFrame limpio y
`split_out` reparte los *group by* grandes entre particiones.

In [ ]:
import dask
import dask.dataframe as dd
M = Medidor("dask")
with M.medir("carga"):
    df = dd.read_csv(DATA, usecols=USE_COLUMNS, dtype=PANDAS_DTYPES, blocksize="64MB",
                     keep_default_na=False, na_values=[""])
    n_total = len(df)
print(f"{n_total:,} filas | {df.npartitions} particiones")
df.head()

In [ ]:
with M.medir("Q1"):
    nulos = df.isna().sum().compute()
    df = df.assign(name=df["name"].str.strip(),
                   population=df["population"].fillna(0).astype("int64"),
                   dem=df["dem"].mask(df["dem"] == -9999),
                   moddate=dd.to_datetime(df["moddate"], format="%Y-%m-%d", errors="coerce"))
guardar(nulos.rename_axis("columna").reset_index(name="nulos"), "dask", "Q1", n=20)

In [ ]:
with M.medir("Q2"):
    dup_id = n_total - df["geonameid"].nunique().compute()
    df = df.drop_duplicates(subset=["geonameid"], split_out=df.npartitions)
    n1 = len(df)
    # hash de la clave compuesta por partición -> groupby distribuido -> solo los grupos repetidos al driver
    k = df[DUP_KEYS].map_partitions(lambda p: pd.util.hash_pandas_object(p, index=False), meta=("k", "uint64"))
    tmp = df[["geonameid"]].assign(k=k)
    g = tmp.groupby("k")["geonameid"].agg(["count", "min"], split_out=8)
    grupos = g[g["count"] > 1].compute()
    en_dup = tmp[tmp["k"].isin(grupos.index.values)].compute().join(grupos["min"], on="k")
    eliminar = en_dup.loc[en_dup["geonameid"] != en_dup["min"], "geonameid"].values
    df = df[~df["geonameid"].isin(eliminar)].persist()
    n2 = len(df)
    q2 = pd.DataFrame({"criterio": ["geonameid", "name+country+lat+lon", "registros_finales"],
                       "valor": [dup_id, n1 - n2, n2]})
guardar(q2, "dask", "Q2")

In [ ]:
with M.medir("Q3"):
    reglas = {"latitud_fuera_rango": ~df["latitude"].between(-90, 90),
              "longitud_fuera_rango": ~df["longitude"].between(-180, 180),
              "poblacion_negativa": df["population"] < 0,
              "pais_nulo": df["country"].isna(),
              "nombre_nulo": df["name"].isna()}
    conteos = dask.compute(*[m.sum() for m in reglas.values()])
    invalido = reglas["latitud_fuera_rango"]
    for m in list(reglas.values())[1:]:
        invalido = invalido | m
    df = df[~invalido]
    q3 = pd.DataFrame({"regla": list(reglas) + ["TOTAL_ELIMINADOS"],
                       "registros": list(conteos) + [n2 - len(df)]})
guardar(q3, "dask", "Q3")

In [ ]:
with M.medir("Q4"):
    es_p = df["fclass"] == "P"
    df = df.assign(
        anio_mod=df["moddate"].dt.year,
        hemisferio=df["latitude"].map_partitions(lambda s: s.ge(0).map({True: "Norte", False: "Sur"}),
                                                 meta=("hemisferio", "object")),
        elevacion_final=df["elevation"].fillna(df["dem"]),
        poblacion_asentamiento=df["population"].where(es_p, 0),
        fclass_desc=df["fclass"].astype("object").map(FCLASS_DESC, meta=("fclass_desc", "object")).fillna("Desconocido"),
        categoria_poblacion=df["population"].map_partitions(
            lambda s: pd.cut(s, bins=POP_BINS, labels=POP_LABELS, right=False).astype(str),
            meta=("categoria_poblacion", "object"))).persist()
    q4 = (df.groupby("categoria_poblacion").size().compute()
            .sort_values(ascending=False).reset_index(name="registros"))
guardar(q4, "dask", "Q4")

In [ ]:
with M.medir("Q5"):
    q5 = (df[(df["country"] == "PE") & (df["fclass"] == "P") & (df["population"] > 0)]
            [["name", "admin1", "population", "elevacion_final", "latitude", "longitude"]]
            .compute().sort_values("population", ascending=False))
print(f"Centros poblados con población en Perú: {len(q5):,}")
guardar(q5.head(20), "dask", "Q5")

In [ ]:
with M.medir("Q6"):
    q6 = (df.groupby("country", observed=True)
            .agg(registros=("geonameid", "count"), poblacion_total=("poblacion_asentamiento", "sum"),
                 elevacion_promedio=("elevacion_final", "mean"))
            .compute().round({"elevacion_promedio": 2})
            .sort_values("poblacion_total", ascending=False).reset_index())
guardar(q6, "dask", "Q6")

In [ ]:
with M.medir("Q7"):
    total = len(df)
    q7 = (df.groupby(["fclass", "fclass_desc"], dropna=False, observed=True).size()
            .compute().reset_index(name="registros").sort_values("registros", ascending=False))
    q7["porcentaje"] = (q7["registros"] / total * 100).round(2)
guardar(q7, "dask", "Q7", n=12)

In [ ]:
with M.medir("Q8"):
    q8 = (df[(df["fclass"] == "P") & (df["population"] > 1_000_000)]
            .groupby("country", observed=True)
            .agg(ciudades_mas_1M=("geonameid", "count"), poblacion_en_esas_ciudades=("population", "sum"))
            .compute().sort_values(["ciudades_mas_1M", "poblacion_en_esas_ciudades"], ascending=False)
            .reset_index())
guardar(q8, "dask", "Q8")

In [ ]:
with M.medir("Q9"):
    sa = df[df["country"].isin(SOUTH_AMERICA) & (df["fclass"] == "P")][["country", "name", "population"]]
    q9 = (sa.groupby("country", observed=True)
            .apply(lambda g: g.nlargest(3, "population"), include_groups=False,
                   meta={"name": "object", "population": "int64"})
            .compute().reset_index(level=0).reset_index(drop=True))
    q9["country"] = q9["country"].astype(str)
    q9["ranking"] = q9.groupby("country")["population"].rank(method="first", ascending=False).astype(int)
    q9 = q9.sort_values(["country", "ranking"])[["country", "ranking", "name", "population"]]
guardar(q9, "dask", "Q9", n=36)

In [ ]:
with M.medir("Q10"):
    q10 = df.groupby("anio_mod").size().compute().sort_index().reset_index(name="registros")
    q10["anio_mod"] = q10["anio_mod"].astype(int)
    q10["pct_acumulado"] = (q10["registros"].cumsum() / q10["registros"].sum() * 100).round(2)
guardar(q10, "dask", "Q10", n=40)
M.cerrar()

In [ ]:
del df, tmp, g, grupos, en_dup, sa, q5, q6, q7, q8, q9, q10
gc.collect();

## 3.3 Modin
Mantiene la **API de pandas**, pero divide el DataFrame en **bloques fila × columna** que procesan en paralelo
los *workers* de **Ray**. Todo vive en memoria: con menos de 16 GB de RAM el *object store* de Ray se llena con
los 13.4 M de registros (en el primer avance el worker moría en Colab). En ese caso esta sección usa
automáticamente la **muestra de 3 M** (semilla 42) y se indica en el benchmark.

In [ ]:
os.environ["MODIN_ENGINE"] = "ray"
import ray
import modin.pandas as mpd
import warnings; warnings.filterwarnings("ignore")

DATA_MODIN = DATA if RAM_GB >= 16 else DATA_3M
print("Modin usará:", DATA_MODIN)
ray.init(object_store_memory=int(min(RAM_GB * 0.3, 8) * 1e9), include_dashboard=False,
         ignore_reinit_error=True, log_to_driver=False)
M = Medidor("modin")
with M.medir("carga"):
    df = mpd.read_csv(DATA_MODIN, usecols=USE_COLUMNS, dtype=PANDAS_DTYPES,
                      keep_default_na=False, na_values=[""])
print(f"{len(df):,} filas")
df.head()

In [ ]:
with M.medir("Q1"):
    nulos = df.isna().sum()._to_pandas()
    df["name"] = df["name"].str.strip()
    df["population"] = df["population"].fillna(0).astype("int64")
    df["dem"] = df["dem"].mask(df["dem"] == -9999)
    df["moddate"] = mpd.to_datetime(df["moddate"], format="%Y-%m-%d", errors="coerce")
guardar(nulos.rename_axis("columna").reset_index(name="nulos"), "modin", "Q1", n=20)

In [ ]:
with M.medir("Q2"):
    dup_id = int(df.duplicated(subset=["geonameid"]).sum())
    df = df.drop_duplicates(subset=["geonameid"])
    n1 = len(df)
    en_dup = df.loc[df.duplicated(subset=DUP_KEYS, keep=False), DUP_KEYS + ["geonameid"]]._to_pandas()
    minimo = en_dup.groupby(DUP_KEYS, observed=True, dropna=False)["geonameid"].transform("min")
    eliminar = en_dup.loc[en_dup["geonameid"] != minimo, "geonameid"].values
    df = df[~df["geonameid"].isin(eliminar)]
    n2 = len(df)
    q2 = pd.DataFrame({"criterio": ["geonameid", "name+country+lat+lon", "registros_finales"],
                       "valor": [dup_id, n1 - n2, n2]})
guardar(q2, "modin", "Q2")

In [ ]:
with M.medir("Q3"):
    reglas = {"latitud_fuera_rango": ~df["latitude"].between(-90, 90),
              "longitud_fuera_rango": ~df["longitude"].between(-180, 180),
              "poblacion_negativa": df["population"] < 0,
              "pais_nulo": df["country"].isna(),
              "nombre_nulo": df["name"].isna()}
    conteos = [int(m.sum()) for m in reglas.values()]
    invalido = reglas["latitud_fuera_rango"]
    for m in list(reglas.values())[1:]:
        invalido = invalido | m
    df = df[~invalido]
    q3 = pd.DataFrame({"regla": list(reglas) + ["TOTAL_ELIMINADOS"], "registros": conteos + [n2 - len(df)]})
guardar(q3, "modin", "Q3")

In [ ]:
with M.medir("Q4"):
    es_p = df["fclass"] == "P"
    df["anio_mod"] = df["moddate"].dt.year
    df["hemisferio"] = (df["latitude"] >= 0).map({True: "Norte", False: "Sur"})
    df["elevacion_final"] = df["elevation"].fillna(df["dem"])
    df["poblacion_asentamiento"] = df["population"].where(es_p, 0)
    df["fclass_desc"] = df["fclass"].astype("object").map(FCLASS_DESC).fillna("Desconocido")
    df["categoria_poblacion"] = mpd.cut(df["population"], bins=POP_BINS, labels=POP_LABELS, right=False).astype(str)
    q4 = (df.groupby("categoria_poblacion").size().sort_values(ascending=False)
            .reset_index(name="registros")._to_pandas())
guardar(q4, "modin", "Q4")

In [ ]:
with M.medir("Q5"):
    q5 = (df[(df["country"] == "PE") & (df["fclass"] == "P") & (df["population"] > 0)]
            [["name", "admin1", "population", "elevacion_final", "latitude", "longitude"]]
            .sort_values("population", ascending=False))
    n_pe = len(q5); q5 = q5.head(20)._to_pandas()
print(f"Centros poblados con población en Perú: {n_pe:,}")
guardar(q5, "modin", "Q5")

In [ ]:
with M.medir("Q6"):
    q6 = (df.groupby("country", observed=True)
            .agg(registros=("geonameid", "count"), poblacion_total=("poblacion_asentamiento", "sum"),
                 elevacion_promedio=("elevacion_final", "mean"))
            .sort_values("poblacion_total", ascending=False).reset_index()._to_pandas()
            .round({"elevacion_promedio": 2}))
guardar(q6, "modin", "Q6")

In [ ]:
with M.medir("Q7"):
    total = len(df)
    q7 = (df.groupby(["fclass", "fclass_desc"], dropna=False, observed=True).size()
            .reset_index(name="registros").sort_values("registros", ascending=False)._to_pandas())
    q7["porcentaje"] = (q7["registros"] / total * 100).round(2)
guardar(q7, "modin", "Q7", n=12)

In [ ]:
with M.medir("Q8"):
    q8 = (df[(df["fclass"] == "P") & (df["population"] > 1_000_000)]
            .groupby("country", observed=True)
            .agg(ciudades_mas_1M=("geonameid", "count"), poblacion_en_esas_ciudades=("population", "sum"))
            .sort_values(["ciudades_mas_1M", "poblacion_en_esas_ciudades"], ascending=False)
            .reset_index()._to_pandas())
guardar(q8, "modin", "Q8")

In [ ]:
with M.medir("Q9"):
    sa = df[df["country"].isin(SOUTH_AMERICA) & (df["fclass"] == "P")][["country", "name", "population"]].copy()
    sa["country"] = sa["country"].astype(str)
    sa["ranking"] = sa.groupby("country")["population"].rank(method="first", ascending=False).astype(int)
    q9 = sa[sa["ranking"] <= 3].sort_values(["country", "ranking"])[["country", "ranking", "name", "population"]]._to_pandas()
guardar(q9, "modin", "Q9", n=36)

In [ ]:
with M.medir("Q10"):
    q10 = df.groupby("anio_mod").size().sort_index().reset_index(name="registros")._to_pandas()
    q10["anio_mod"] = q10["anio_mod"].astype(int)
    q10["pct_acumulado"] = (q10["registros"].cumsum() / q10["registros"].sum() * 100).round(2)
guardar(q10, "modin", "Q10", n=40)
M.cerrar()

In [ ]:
del df, sa, en_dup, q5, q6, q7, q8, q9, q10
gc.collect()
ray.shutdown()

## 3.4 Apache Spark
DataFrames de Spark sobre la JVM: el plan se optimiza con **Catalyst** y se divide en *stages* y *tasks* por
partición. En Dataproc la sesión corre sobre **YARN**, repartida entre los 2 workers, y lee el CSV directamente
del bucket (`gs://`). Q9 y Q10 usan **funciones de ventana**. Al final se escribe la capa *processed* en
Parquet particionado por `fclass`.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F, types as T

spark = (SparkSession.builder.appName("GeoNames Big Data - UTEC")
         .config("spark.sql.shuffle.partitions", "32")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.driver.memory", "4g")          # solo aplica en modo local
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", spark.sparkContext.master)

SCHEMA_SP = T.StructType([T.StructField(c, t) for c, t in [
    ("geonameid", T.LongType()), ("name", T.StringType()), ("asciiname", T.StringType()),
    ("alternatenames", T.StringType()), ("latitude", T.DoubleType()), ("longitude", T.DoubleType()),
    ("fclass", T.StringType()), ("fcode", T.StringType()), ("country", T.StringType()),
    ("cc2", T.StringType()), ("admin1", T.StringType()), ("admin2", T.StringType()),
    ("admin3", T.StringType()), ("admin4", T.StringType()), ("population", T.LongType()),
    ("elevation", T.DoubleType()), ("dem", T.LongType()), ("timezone", T.StringType()),
    ("moddate", T.StringType())]])

ENTRADA_SPARK = GCS_RAW if EN_DATAPROC else DATA
M = Medidor("spark")
with M.medir("carga"):
    sdf = (spark.read.option("header", True).option("quote", '"').option("escape", '"')
                .schema(SCHEMA_SP).csv(ENTRADA_SPARK).select(*USE_COLUMNS).cache())
    n0 = sdf.count()
print(f"{n0:,} filas | {sdf.rdd.getNumPartitions()} particiones | origen: {ENTRADA_SPARK}")
sdf.show(5)

In [ ]:
with M.medir("Q1"):
    nulos = sdf.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in sdf.columns]).toPandas().T.reset_index()
    nulos.columns = ["columna", "nulos"]
    sdf = (sdf.withColumn("name", F.trim("name"))
              .withColumn("population", F.coalesce("population", F.lit(0)))
              .withColumn("dem", F.when(F.col("dem") == -9999, None).otherwise(F.col("dem")))
              .withColumn("moddate", F.to_date("moddate", "yyyy-MM-dd")))
guardar(nulos, "spark", "Q1", n=20)

In [ ]:
with M.medir("Q2"):
    dup_id = n0 - sdf.select("geonameid").distinct().count()
    sdf = sdf.dropDuplicates(["geonameid"])
    n1 = sdf.count()
    conservar = sdf.groupBy(*DUP_KEYS).agg(F.min("geonameid").alias("geonameid"))
    sdf = sdf.join(conservar.select("geonameid"), on="geonameid", how="left_semi").cache()
    n2 = sdf.count()
    q2 = pd.DataFrame({"criterio": ["geonameid", "name+country+lat+lon", "registros_finales"],
                       "valor": [dup_id, n1 - n2, n2]})
guardar(q2, "spark", "Q2")

In [ ]:
with M.medir("Q3"):
    reglas = {"latitud_fuera_rango": ~F.col("latitude").between(-90, 90),
              "longitud_fuera_rango": ~F.col("longitude").between(-180, 180),
              "poblacion_negativa": F.col("population") < 0,
              "pais_nulo": F.col("country").isNull(),
              "nombre_nulo": F.col("name").isNull()}
    conteos = sdf.select([F.sum(F.coalesce(e, F.lit(False)).cast("int")).alias(k)
                          for k, e in reglas.items()]).first().asDict()
    invalido = None
    for e in reglas.values():
        e = F.coalesce(e, F.lit(False))
        invalido = e if invalido is None else (invalido | e)
    sdf = sdf.filter(~invalido)
    n3 = sdf.count()
    q3 = pd.DataFrame({"regla": list(conteos) + ["TOTAL_ELIMINADOS"],
                       "registros": list(conteos.values()) + [n2 - n3]})
guardar(q3, "spark", "Q3")

In [ ]:
with M.medir("Q4"):
    mapa = F.create_map([F.lit(x) for kv in FCLASS_DESC.items() for x in kv])
    pop = F.col("population")
    sdf = (sdf.withColumn("anio_mod", F.year("moddate"))
              .withColumn("hemisferio", F.when(F.col("latitude") >= 0, "Norte").otherwise("Sur"))
              .withColumn("elevacion_final", F.coalesce(F.col("elevation"), F.col("dem").cast("double")))
              .withColumn("poblacion_asentamiento", F.when(F.col("fclass") == "P", pop).otherwise(F.lit(0)))
              .withColumn("fclass_desc", F.coalesce(mapa[F.col("fclass")], F.lit("Desconocido")))
              .withColumn("categoria_poblacion",
                          F.when(pop <= 0, POP_LABELS[0]).when(pop < 1_000, POP_LABELS[1])
                           .when(pop < 100_000, POP_LABELS[2]).when(pop < 1_000_000, POP_LABELS[3])
                           .otherwise(POP_LABELS[4]))
              .cache())
    q4 = sdf.groupBy("categoria_poblacion").agg(F.count("*").alias("registros")).orderBy(F.desc("registros")).toPandas()
guardar(q4, "spark", "Q4")

In [ ]:
with M.medir("Q5"):
    pe = sdf.filter((F.col("country") == "PE") & (F.col("fclass") == "P") & (F.col("population") > 0))
    n_pe = pe.count()
    q5 = (pe.orderBy(F.desc("population"))
            .select("name", "admin1", "population", "elevacion_final", "latitude", "longitude")
            .limit(20).toPandas())
print(f"Centros poblados con población en Perú: {n_pe:,}")
guardar(q5, "spark", "Q5")

In [ ]:
with M.medir("Q6"):
    q6 = (sdf.groupBy("country")
             .agg(F.count("*").alias("registros"),
                  F.sum("poblacion_asentamiento").alias("poblacion_total"),
                  F.round(F.avg("elevacion_final"), 2).alias("elevacion_promedio"))
             .orderBy(F.desc("poblacion_total")).toPandas())
guardar(q6, "spark", "Q6")

In [ ]:
with M.medir("Q7"):
    total = sdf.count()
    q7 = (sdf.groupBy("fclass", "fclass_desc").agg(F.count("*").alias("registros"))
             .withColumn("porcentaje", F.round(F.col("registros") / total * 100, 2))
             .orderBy(F.desc("registros")).toPandas())
guardar(q7, "spark", "Q7", n=12)

In [ ]:
with M.medir("Q8"):
    q8 = (sdf.filter((F.col("fclass") == "P") & (F.col("population") > 1_000_000))
             .groupBy("country")
             .agg(F.count("*").alias("ciudades_mas_1M"), F.sum("population").alias("poblacion_en_esas_ciudades"))
             .orderBy(F.desc("ciudades_mas_1M"), F.desc("poblacion_en_esas_ciudades")).toPandas())
guardar(q8, "spark", "Q8")

In [ ]:
with M.medir("Q9"):
    w = Window.partitionBy("country").orderBy(F.desc("population"))
    q9 = (sdf.filter(F.col("country").isin(SOUTH_AMERICA) & (F.col("fclass") == "P"))
             .withColumn("ranking", F.row_number().over(w))
             .filter(F.col("ranking") <= 3)
             .select("country", "ranking", "name", "population")
             .orderBy("country", "ranking").toPandas())
guardar(q9, "spark", "Q9", n=36)

In [ ]:
with M.medir("Q10"):
    w_acum = Window.orderBy("anio_mod").rowsBetween(Window.unboundedPreceding, 0)
    q10 = (sdf.filter(F.col("anio_mod").isNotNull())
              .groupBy("anio_mod").agg(F.count("*").alias("registros"))
              .withColumn("pct_acumulado", F.round(F.sum("registros").over(w_acum)
                                                   / F.sum("registros").over(Window.partitionBy()) * 100, 2))
              .orderBy("anio_mod").toPandas())
guardar(q10, "spark", "Q10", n=40)
M.cerrar()

In [ ]:
# Capa PROCESSED del Data Lake: Parquet particionado por clase de entidad
if EN_DATAPROC and ESCRIBIR_PROCESSED:
    t0 = time.perf_counter()
    sdf.write.mode("overwrite").partitionBy("fclass").parquet(f"{GCS_PROCESSED}/spark_parquet")
    print(f"Parquet escrito en {GCS_PROCESSED}/spark_parquet en {time.perf_counter() - t0:.1f} s")
    sh(f"gcloud storage ls {GCS_PROCESSED}/spark_parquet/")
    sh(f"gcloud storage du -s -r {GCS_PROCESSED}/spark_parquet")
sdf.unpersist()

## 4. Comparación de rendimiento

In [ ]:
bench = pd.DataFrame(TIEMPOS).T
bench["ram_pico_gb"] = pd.Series(RAM_PICO)
bench["dataset"] = ["muestra 3M" if (fw == "modin" and DATA_MODIN != DATA) else "completo" for fw in bench.index]
cols = ["dataset", "carga"] + list(QUERIES) + ["total_consultas", "ram_pico_gb"]
bench = bench[cols]
bench.to_csv(f"{OUT}/benchmark.csv")
bench

In [ ]:
colores = {"polars": "#2a6fdb", "dask": "#e8871e", "modin": "#8e44ad", "spark": "#e03c31"}
b = bench.copy(); c = [colores[f] for f in b.index]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, col, tit in zip(ax, ["carga", "total_consultas", "ram_pico_gb"],
                       ["Carga del CSV (s)", "10 consultas (s)", "RAM pico (GB)"]):
    a.bar(b.index, b[col].astype(float), color=c); a.set_title(tit)
    a.spines[["top", "right"]].set_visible(False)
    for i, v in enumerate(b[col].astype(float)):
        a.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/benchmark_resumen.png", dpi=150); plt.show()

fig, a = plt.subplots(figsize=(13, 4.5)); ancho = 0.2
for i, fw in enumerate(b.index):
    a.bar([j + i * ancho for j in range(10)], b.loc[fw, list(QUERIES)].astype(float), ancho,
          label=fw, color=colores[fw])
a.set_xticks([j + 1.5 * ancho for j in range(10)]); a.set_xticklabels(list(QUERIES))
a.set_ylabel("segundos"); a.set_title("Tiempo por consulta"); a.legend(frameon=False)
a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT}/benchmark_por_consulta.png", dpi=150); plt.show()

### Verificación: los frameworks devuelven el mismo resultado
Se ordena cada resultado, se normalizan los decimales y se calcula un hash MD5 por consulta. En Q9 se excluyen
los empates de población 0, cuyo orden no está definido. Si Modin corrió con la muestra de 3 M, se compara aparte.

In [ ]:
def firma(fw, q):
    d = pd.read_csv(f"{OUT}/{fw}/{q}.csv", keep_default_na=False, na_values=[""])
    if q == "Q9":
        d = d[d["population"] > 0]
    d = d.fillna("NA").sort_values(list(d.columns)).reset_index(drop=True).round(1)
    return hashlib.md5(d.to_csv(index=False).encode()).hexdigest()[:8]

comparables = [fw for fw in TIEMPOS if not (fw == "modin" and DATA_MODIN != DATA)]
ver = pd.DataFrame({fw: [firma(fw, q) for q in QUERIES] for fw in comparables}, index=list(QUERIES))
ver["iguales"] = ver.nunique(axis=1) == 1
ver

## 5. Evidencias de Google Cloud Platform
Bucket del Data Lake y clúster Dataproc (1 master + 2 workers `n4d-standard-2`, imagen 3.0, `us-east4-b`).

In [ ]:
sh(f"gcloud storage buckets describe gs://{BUCKET} --format='table(name,location,default_storage_class,time_created)'")
sh(f"gcloud storage ls -r -l 'gs://{BUCKET}/raw/**'")
sh(f"gcloud storage ls gs://{BUCKET}/")
sh(f"gcloud storage du -s -r gs://{BUCKET}")

In [ ]:
sh(f"gcloud dataproc clusters describe {CLUSTER} --region={REGION} "
   "--format='yaml(clusterName,config.masterConfig.machineTypeUri,config.masterConfig.numInstances,"
   "config.workerConfig.machineTypeUri,config.workerConfig.numInstances,config.softwareConfig.imageVersion,status.state)'")
sh("yarn node -list 2>/dev/null | head -10")
sh("hdfs dfsadmin -report 2>/dev/null | head -25")

## 6. Hadoop MapReduce en Google Cloud Dataproc
Se usan programas de `hadoop-mapreduce-examples.jar` (sin `wordcount` ni `grep`) sobre datos del propio GeoNames
guardados en **HDFS**, y los resultados se escriben también en **HDFS**:

| Programa | Objetivo | Entrada HDFS | Salida HDFS |
|---|---|---|---|
| `wordmean` | Longitud media de las palabras de los nombres geográficos | `/data/input/nombres` | `/data/output/wordmean` |
| `wordmedian` | Longitud mediana de esas palabras (histograma longitud → frecuencia) | `/data/input/nombres` | `/data/output/wordmedian` |
| `secondarysort` | Elevaciones ordenadas dentro de cada franja de latitud (clave compuesta, orden secundario) | `/data/input/secondarysort` | `/data/output/secondarysort` |
| `terasort` (extra) | Ordenamiento global distribuido teragen → terasort → teravalidate | generado | `/data/output/terasort` |

> Esta sección requiere que el notebook corra **en el master del clúster** (Jupyter de Dataproc).

In [ ]:
JAR = "/usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar"
PUEDE_MR = EJECUTAR_MAPREDUCE and EN_DATAPROC
print("Ejecutar MapReduce:", PUEDE_MR)
if PUEDE_MR:
    sh(f"hadoop jar {JAR} 2>&1 | head -45")        # programas disponibles en el JAR

### 6.1 Preparación de las entradas y carga a HDFS
- `nombres.txt`: un nombre geográfico por línea (para `wordmean` y `wordmedian`).
- `lat_dem.txt`: pares `<franja_latitud> <elevación_DEM>` en enteros; se omiten los DEM = −9999 (para `secondarysort`).

In [ ]:
import csv, math
def preparar_entradas(csv_path, dir_salida):
    csv.field_size_limit(10**9)
    n = m = 0
    with open(csv_path, encoding="utf-8", newline="") as fin, \
         open(f"{dir_salida}/nombres.txt", "w", encoding="utf-8") as f_nom, \
         open(f"{dir_salida}/lat_dem.txt", "w") as f_sec:
        for fila in csv.DictReader(fin):
            n += 1
            f_nom.write(fila["name"].replace("\t", " ") + "\n")
            dem = fila["dem"]
            if dem and dem != "-9999":
                f_sec.write(f"{math.floor(float(fila['latitude']))} {int(dem)}\n"); m += 1
    print(f"nombres.txt: {n:,} líneas | lat_dem.txt: {m:,} pares")

if PUEDE_MR:
    preparar_entradas(DATA, DATA_DIR)
    sh("hdfs dfs -mkdir -p /data/input/nombres /data/input/secondarysort /data/input/raw /data/output")
    sh(f"hdfs dfs -put -f {DATA_DIR}/nombres.txt /data/input/nombres/")
    sh(f"hdfs dfs -put -f {DATA_DIR}/lat_dem.txt /data/input/secondarysort/")
    sh(f"hadoop distcp -overwrite {GCS_RAW} /data/input/raw/")      # CSV crudo del Data Lake a HDFS
    sh("hdfs dfs -ls -R /data/input")
    sh("hdfs dfs -du -h /data/input")
    sh("hdfs fsck /data/input -files -blocks | tail -20")
    sh("hdfs dfs -rm -r -f /data/output/wordmean /data/output/wordmedian /data/output/secondarysort")

### 6.2 Programa 1: `wordmean`
El mapper emite, por cada palabra, `count = 1` y `length = longitud`; el reducer (también combiner) suma ambos y
el driver calcula `length / count`.

In [ ]:
if PUEDE_MR:
    sh(f"time hadoop jar {JAR} wordmean /data/input/nombres /data/output/wordmean")
    sh("hdfs dfs -ls /data/output/wordmean")
    sh("hdfs dfs -cat /data/output/wordmean/part-r-00000")

### 6.3 Programa 2: `wordmedian`
El mapper emite `(longitud, 1)`, el reducer arma el histograma de frecuencias y el driver recorre el histograma
hasta la posición central.

In [ ]:
if PUEDE_MR:
    sh(f"time hadoop jar {JAR} wordmedian /data/input/nombres /data/output/wordmedian")
    sh("hdfs dfs -ls /data/output/wordmedian")
    sh("hdfs dfs -cat /data/output/wordmedian/part-r-00000 | sort -n | head -30")

### 6.4 Programa 3: `secondarysort`
Clave compuesta `IntPair(franja, elevación)`: se particiona y agrupa por la franja y se ordena por ambos valores,
así cada reducer recibe las elevaciones de una franja ya ordenadas (primer valor = mínimo, último = máximo).

In [ ]:
if PUEDE_MR:
    sh(f"time hadoop jar {JAR} secondarysort /data/input/secondarysort /data/output/secondarysort")
    sh("hdfs dfs -ls /data/output/secondarysort")
    sh("hdfs dfs -du -h /data/output/secondarysort")
    print("\nFranja -12 (Lima): primeras y últimas elevaciones ordenadas")
    sh("hdfs dfs -cat '/data/output/secondarysort/part-r-*' | awk -F'\\t' '$1==\"-12\"' | head -5")
    sh("hdfs dfs -cat '/data/output/secondarysort/part-r-*' | awk -F'\\t' '$1==\"-12\"' | tail -5")
    print("\nElevación máxima por franja (top 10)")
    sh("hdfs dfs -cat '/data/output/secondarysort/part-r-*' | grep -v '^---' "
       "| awk -F'\\t' '{m[$1]=$2} END {for (k in m) print k\"\\t\"m[k]}' | sort -k2 -n -r | head -10")

### 6.5 (Extra) `terasort`

In [ ]:
if PUEDE_MR and EJECUTAR_TERASORT:
    sh("hdfs dfs -rm -r -f /data/output/teragen /data/output/terasort /data/output/teravalidate")
    sh(f"time hadoop jar {JAR} teragen 10000000 /data/output/teragen")
    sh(f"time hadoop jar {JAR} terasort /data/output/teragen /data/output/terasort")
    sh(f"time hadoop jar {JAR} teravalidate /data/output/terasort /data/output/teravalidate")
    sh("hdfs dfs -cat /data/output/teravalidate/part-r-00000")

### 6.6 Persistencia de los resultados en el Data Lake y valores de control

In [ ]:
if PUEDE_MR:
    sh(f"hadoop distcp -overwrite /data/output/wordmean /data/output/wordmedian /data/output/secondarysort "
       f"gs://{BUCKET}/mapreduce/output/")
    sh(f"gcloud storage ls -r gs://{BUCKET}/mapreduce/")
    sh("mapred job -list all 2>/dev/null | tail -10")

In [ ]:
# Valores de control calculados con Polars sobre los mismos datos (para validar la salida de MapReduce)
nom = pl.scan_csv(DATA, infer_schema_length=0).select("name", "latitude", "dem").collect()
lens = (nom.select(pl.col("name").fill_null("").str.split(" ").alias("w")).explode("w")
           .filter(pl.col("w").str.len_chars() > 0).select(pl.col("w").str.len_chars())).to_series()
pares = (nom.filter(pl.col("dem") != "-9999")
            .select(pl.col("latitude").cast(pl.Float64).floor().cast(pl.Int32).alias("franja"),
                    pl.col("dem").cast(pl.Int32).alias("dem"))
            .group_by("franja").agg(pl.len().alias("n"), pl.col("dem").min().alias("min"),
                                    pl.col("dem").max().alias("max")))
print(f"wordmean   -> palabras: {len(lens):,} | longitud media: {lens.mean():.4f}")
print(f"wordmedian -> mediana: {int(lens.sort()[(len(lens) - 1) // 2])}")
print("secondarysort -> franja -12:", pares.filter(pl.col("franja") == -12).to_dicts())
del nom, lens; gc.collect();

## 7. Indicadores para la toma de decisiones

In [ ]:
print("Población en asentamientos por país (top 10)")
display(pd.read_csv(f"{OUT}/polars/Q6.csv").head(10))
print("Ciudades de más de 1 M de habitantes por país (top 10)")
display(pd.read_csv(f"{OUT}/polars/Q8.csv").head(10))
print("Top 3 ciudades por país de Sudamérica")
display(pd.read_csv(f"{OUT}/polars/Q9.csv", keep_default_na=False))